# Курсовой проект, MDS

## Проектная группа:
- Даниил Нифанин
- Данил Николаев
- Дмитрий Тамендаров
- Мария Вичентиевич
- Светлана Максимова

## Общая информация

Стек: pandas, numpy, matplotlib, seaborn  
Источники: NetflixShows.xlsx, OMDB API, открытые датасеты Kaggle


## Введение

Netflix, это международный стримминговый сервис фильмов и сериалов. У сервиса более 300 млн зрителей и у каждого есть предпочтения. Одной из задач является оценка успешности шоу и фильмов, которые будут включены в подписку. Для прогноза успешности продукта нужно уметь делать выводы по описанию фильма, шоу. Для этого компания непрерывно собирает обратную связь от пользователей и хранит ингформацию о каждом шоу. В данной работу предпринята попытка предсказать успешность шоу или фильма по открытой информации о 1000 шоу по состоянию на 11.06.2017 - 1000 Netflix Shows.  
**Цель:** По характеристикам шоу/фильма предсказать успешность в регионах мира.  
**Задачи:**
- предобработка основного датасета
- Предложить новые признаки на основе имеющихся
- обогатить датасет с помощью внешних источников
- демографический анализ
- анализ шоу
- анализ количества и качества оценок
- прогноз на основе проведенного анализа

## Описание датасета 1000 NetflixShows

### Описание признаков

- title - название шоу.
- rating - рейтинг шоу. Например: G, PG, TV-14, TV-MA.
- ratingLevel - описание рейтинговой группы и особенностей шоу.
- ratingDescription - рейтинг шоу, закодированный числом.
- release year - год выпуска
user rating score - оценка пользователей.
- user rating size - общий рейтинг пользователей.

### Требования к проекту

В качестве результата выполнения курсового проекта ваша команда должна получить презентацию и защитить ее перед комиссией.

Оформление презентации остаётся полностью на ваше усмотрение, но помните, что результат должен быть релевантен для демонстрации бизнес-заказчику — комиссию, принимающую вашу работу, правильнее всего воспринимать именно в таком качестве. Например, вставлять в презентацию строчки кода или злоупотреблять скринами блокнота не рекомендуется.

С точки зрения концепции выполнения проекта — вам необходимо принять на себя роль аналитиков: провести работу над признаками, исследовать информацию, содержащуюся в датасете,  выявить тенденции, тренды, факты из данных, а также, конечно, презентовать всё это в понятном виде.

Фактически, можно воспринимать этот проект в следующем ключе: к вам пришел некий бизнес-заказчик — например, это может быть непосредственно представитель Netflix, которые хотят улучшить какие-то процессы; или какая-то компания, которая хочет снять какой-то новый проект и/или продать какой-то свой уже готовый продукт Netflix'у; или это может быть некая компания конкурент Netflix'a, которая заинтересована в общем исследовании рынка; или абсолютно любой другой стейхолдер в рамках данной отрасли — и вот этот заказчик просит вас проанализировать данные и извлечь на основе них какие-то полезные и значимые бизнес-инсайты для него. Разумеется, чем глубже, чем осмысленнее и чем нетривиальнее будут эти выводы, тем больше они вам заплатят :)


## Предобработка данных

### Импортирование датасета и библиотек для работы

In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("API_KEY")
data = pd.read_excel("NetflixShows.xlsx")
del data['ratingDescription']
data


,title,rating,ratingLevel,release year,user rating score,user rating size
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,80
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,80
...,...,...,...,...,...,...
995,The BFG,PG,"for action/peril, some scary moments and brief...",2016,97.0,80
996,The Secret Life of Pets,PG,for action and some rude humor,2016,NaN,81
997,Precious Puppies,TV-G,Suitable for all ages.,2003,NaN,82
998,Beary Tales,TV-G,Suitable for all ages.,2013,NaN,82


### Обработка дубликатов

#### Удаление полных дубликатов

In [2]:
print("Дубликатов:", data.duplicated().sum())
print("Доля:", data.duplicated().mean())
print("Среднее количество повторений:", len(data) / len(data.drop_duplicates()))


Дубликатов: 500
Доля: 0.5
Среднее количество повторений: 2.0


Полностью дублирующихся строк 500 - их можно удалить, не опасаясь потери информации. Доля дубликатов составляет 50% от общего объема и среднее количество повторений равное 2-м указывает на сильное дублирование информации. Это возможно при некорректной загрузке данных, ошибка могла произойти либо при объеденении данных, либо выгрузка была произведена дважды.
Поскольку 500 строк являются полностью идентичными - их можно удалить, не опасаясь потери информации.

In [3]:
# Сгруппируем дубликаты по возрастным рейтингам для промотра того, какоц рейтинг имеет наибольшее количество дубликатов
# Источник: https://thetahat.ru/courses/python/08$0
pd.crosstab(data['rating'],
            data.duplicated(),
            colnames=None,
            margins=True
            ).set_axis(['unique', 'duplicates', 'all'],
                       axis=1
                       )

,unique,duplicates,all
rating,,,
G,53,85,138
NR,10,4,14
PG,76,94,170
PG-13,12,3,15
R,14,5,19
TV-14,106,128,234
TV-G,29,23,52
TV-MA,82,66,148
TV-PG,33,26,59


При просмотре дубликатов по группам больше всего наблюдается TV-14, PG, G

In [4]:
# Удалим полные дубликаты
data_no_duplicates = data.drop_duplicates()
data_no_duplicates.info()


<class 'pandas.DataFrame'>
Index: 500 entries, 0 to 998
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              500 non-null    object 
 1   rating             500 non-null    str    
 2   ratingLevel        467 non-null    str    
 3   release year       500 non-null    int64  
 4   user rating score  256 non-null    float64
 5   user rating size   500 non-null    int64  
dtypes: float64(1), int64(2), object(1), str(2)
memory usage: 27.3+ KB


#### Рассмотр дубликатов по title

Далее рассмотрим дубликаты по признаку title

In [5]:
data_no_duplicates[data_no_duplicates.duplicated('title', keep=False)].sort_values(by='title')


,title,rating,ratingLevel,release year,user rating score,user rating size
167,Bordertown,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,86.0,80
449,Bordertown,TV-MA,For mature audiences. May not be suitable for...,2016,NaN,82
568,Goosebumps,TV-Y7,Suitable for children ages 7 and older,1998,88.0,80
632,Goosebumps,PG,"scary and intense creature action and images, ...",2015,90.0,80
151,Skins,TV-MA,For mature audiences. May not be suitable for...,2013,NaN,82
181,Skins,TV-MA,NaN,2017,NaN,82
504,Star Wars: The Clone Wars,PG,"sci-fi action violence throughout, brief langu...",2008,57.0,80
512,Star Wars: The Clone Wars,TV-PG,Parental guidance suggested. May not be suitab...,2014,93.0,80


Объекты отличаются по признакам rating и/или release year, а также по оценкам, что означает, что это разные шоу и их нужно оставить.

In [6]:
df = data_no_duplicates.reset_index()


### Обработка пустых значений

Далее необходимо обработать пустые значения по каждому признаку

In [7]:
df.isna().sum() / df.shape[0]


index                0.000
title                0.000
rating               0.000
ratingLevel          0.066
release year         0.000
user rating score    0.488
user rating size     0.000
dtype: float64

Видим пропуски в ratingLevel и user rating score.
1. Чтобы заполнить пропуски `ratingLevel` изучим его содержимое и позже заполним модой по соответствующему rating, так как данные признаки связаны: rating - сертификат контент рейтинга, ratingLevel - описание данного возрастного рейтинга и сцен присутствующих в фильме.
2. Пропуск по `user rating score` не получится обработать средствами данного датафрейма, так как пропуск значимый(>0.3)

In [8]:
df_rating_nan = df.copy()
df_rating_nan['urs_nan'] = df_rating_nan['user rating score'].isnull()
df_rating_nan.groupby('rating')['urs_nan'].agg(
    nan=('sum'), all=('count'),
    percent=(lambda x: f'{x.sum() * 100 / x.count():.2f}%')
    )


,nan,all,percent
rating,,,
G,34,53,64.15%
NR,8,10,80.00%
PG,28,76,36.84%
PG-13,3,12,25.00%
R,7,14,50.00%
TV-14,29,106,27.36%
TV-G,18,29,62.07%
TV-MA,42,82,51.22%
TV-PG,12,33,36.36%


Значения пропусков в `user rating score` в разных рейтинговых группах отличаюстя. Но тем не менее, в каждой рейтнговой группе доля пропусков является значимой и их заполнение внесет искажение в будущий анализ

#### Работа с рейтинговыми группами

In [9]:
print(f'количество уникальных рейтинговых групп: {df['rating'].nunique()}')
print(f'список уникальных рейтинговых групп: {', '.join(df['rating'].unique())}')


количество уникальных рейтинговых групп: 13
список уникальных рейтинговых групп: PG-13, R, TV-14, TV-PG, TV-MA, TV-Y, NR, TV-Y7-FV, UR, PG, TV-G, G, TV-Y7


Проверим зависимость между рейтинговыми группами и описанием группы, есть ли в них отличия или соответствие идет 1 к 1 - для одной группы одно описание.

In [10]:
df.groupby('rating')['ratingLevel'].agg(num_unique='nunique', all='count')


,num_unique,all
rating,,
G,1,52
NR,1,7
PG,63,76
PG-13,12,12
R,14,14
TV-14,2,101
TV-G,1,29
TV-MA,1,60
TV-PG,1,31


на основе сводной таблицы выше можно сделать вывод, что большинство описаний имеют соответсвие 1 к 1. Но в группах PG, PG-13, R описаний много на одну группу, обработаем данные описания. Также в группе TV-14 есть два разных описания. Начнем с TV-14


In [11]:
df[df['rating'] == 'TV-14'].groupby(['rating', 'ratingLevel']).size()


rating  ratingLevel                                                                  
TV-14   Parents strongly cautioned. May be unsuitable for children ages 14 and under.    100
        dialogue, language, sexual situations and violence                                 1
dtype: int64

In [12]:
df[df['ratingLevel'] == 'dialogue, language, sexual situations and violence']


,index,title,rating,ratingLevel,release year,user rating score,user rating size
233,412,Hawaii Five-0,TV-14,"dialogue, language, sexual situations and viol...",2016,96.0,80


Картина Hawaii Five-0 оценена рейтингом TV-14 и имеет отличное описание от всех остальных, в ней описываются сцены, из-за которых рейтинг оказался таким. Разделим описание на два вида. Одно с описанием рейтинга, второе с описанием сцен в фильме.

In [13]:
df['sceneDesc'] = None
df.loc[df['title'] == 'Hawaii Five-0', 'sceneDesc'] = df.loc[df['title'] == 'Hawaii Five-0', 'ratingLevel']
df.loc[df['title'] == 'Hawaii Five-0', 'ratingLevel'] = 'Parents strongly cautioned. May be unsuitable for children ages 14 and under.'
df[df['title'] == 'Hawaii Five-0']

,index,title,rating,ratingLevel,release year,user rating score,user rating size,sceneDesc
233,412,Hawaii Five-0,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,96.0,80,"dialogue, language, sexual situations and viol..."


In [14]:
df[df['rating'] == 'TV-14'].groupby(['rating', 'ratingLevel']).size()

rating  ratingLevel                                                                  
TV-14   Parents strongly cautioned. May be unsuitable for children ages 14 and under.    101
dtype: int64

In [15]:
df[df['rating'] == 'R'].groupby(['rating', 'ratingLevel']).size()

rating  ratingLevel                                                                    
R       Restricted. May be inappropriate for children 17 and under.                        1
        bloody war violence, language throughout and some sexual material                  1
        language and brief violence                                                        1
        language, drug content, sexuality/nudity, and some violence-all involving teens    1
        language, some drug use, violence and partial nudity                               1
        pervasive drug content and language, some violence and sexuality                   1
        pervasive language, some sexual material, violence and drug use                    1
        pervasive sexual content, brief graphic nudity, language and some drug use         1
        some sexual material                                                               1
        some sexual material, and language throughout                      

В рейтинге R всего 14 записей и все они разные. Но есть одна запись с описанием рейтинга, запишем в ratingLevel данную запись, а описаание сцен перенесем в соответствующий признак.

In [16]:
df.loc[(df['rating'] == 'R') & (df['ratingLevel'] != 'Restricted. May be inappropriate for children 17 and under.'), 'sceneDesc'] = \
df[(df['rating'] == 'R') & (df['ratingLevel'] != 'Restricted. May be inappropriate for children 17 and under.')]['ratingLevel']

df.loc[(df['rating'] == 'R') & (df['ratingLevel'] != 'Restricted. May be inappropriate for children 17 and under.'), 'ratingLevel'] = \
'Restricted. May be inappropriate for children 17 and under.'
df[df['rating'] == 'R'].sample(1)

,index,title,rating,ratingLevel,release year,user rating score,user rating size,sceneDesc
35,35,Hyena Road,R,Restricted. May be inappropriate for children ...,2015,NaN,82,"bloody war violence, language throughout and s..."


Даллее обработаем рейтинг PG-13

In [17]:
df.loc[(df['rating'] == 'PG-13') & (df['ratingLevel'] != 'Parents strongly cautioned. May be inappropriate for children under 13.'), 'sceneDesc'] = \
df[(df['rating'] == 'PG-13') & (df['ratingLevel'] != 'Parents strongly cautioned. May be inappropriate for children under 13.')]['ratingLevel']

df.loc[(df['rating'] == 'PG-13') & (df['ratingLevel'] != 'Parents strongly cautioned. May be inappropriate for children under 13.'), 'ratingLevel'] = \
'Parents strongly cautioned. May be inappropriate for children under 13.'
df[df['rating'] == 'PG-13'].sample(1)

,index,title,rating,ratingLevel,release year,user rating score,user rating size,sceneDesc
214,372,Unconditional,PG-13,Parents strongly cautioned. May be inappropria...,2012,NaN,82,some violent content and mature thematic elements


Далее преобразуем самый обширный рейтинг PG

In [18]:
df[(df['rating'] == 'PG') & (df['ratingLevel'].str.contains('parent', case=False))].head()

,index,title,rating,ratingLevel,release year,user rating score,user rating size,sceneDesc
148,200,Nancy Drew,PG,Parental guidance suggested. May not be suitab...,2007,NaN,82,None
165,245,The Matchbreaker,PG,Parental guidance suggested. May not be suitab...,2016,79.0,80,None
170,252,Grease,PG,Parental guidance suggested. May not be suitab...,1978,86.0,80,None
293,529,Agent F.O.X.,PG,Parental guidance suggested. May not be suitab...,2014,NaN,82,None
395,700,Cool Runnings,PG,Parental guidance suggested. May not be suitab...,1993,81.0,80,None


В PG также есть описание рейтинга, проделаем тоже самое, что и в предыдущих шагах

In [19]:
df.loc[(df['rating'] == 'PG') & (df['ratingLevel'] != 'Parental guidance suggested. May not be suitable for children.'), 'sceneDesc'] = \
df[(df['rating'] == 'PG') & (df['ratingLevel'] != 'Parental guidance suggested. May not be suitable for children.')]['ratingLevel']

df.loc[(df['rating'] == 'PG') & (df['ratingLevel'] != 'Parental guidance suggested. May not be suitable for children.'), 'ratingLevel'] = \
'Parental guidance suggested. May not be suitable for children.'

df[df['rating'] == 'PG'].sample(1)

,index,title,rating,ratingLevel,release year,user rating score,user rating size,sceneDesc
176,258,Coraline,PG,Parental guidance suggested. May not be suitab...,2009,95.0,80,"thematic elements, scary images, some language..."


После приведения ratingLevel к одному виду можем заполнить пустые значения, значениями из соответсвующего рейтинга

In [20]:
moda = df.groupby('rating')['ratingLevel'].transform(lambda x: x.mode()[0])
df['ratingLevel'] = df['ratingLevel'].fillna(moda)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   index              500 non-null    int64  
 1   title              500 non-null    object 
 2   rating             500 non-null    str    
 3   ratingLevel        500 non-null    str    
 4   release year       500 non-null    int64  
 5   user rating score  256 non-null    float64
 6   user rating size   500 non-null    int64  
 7   sceneDesc          89 non-null     object 
dtypes: float64(1), int64(3), object(2), str(2)
memory usage: 31.4+ KB


### Feature engineering

#### Разделение объектов по типам на основе возрастного рейтнга

После изучения рейтинговых групп, группы, которые начинаются с TV являются телешоу (телесериалами) и на основе этих данных можно вывести информацию о типе объекта. Рейтинги TV-14, TV-PG, TV-MA, TV-Y, TV-Y7-FV, TV-G, TV-Y7 относятся к группе TV. Рейтинги: PG-13, R, PG, G к movie. Для них определим типы tv и movie. Ретинги NR и UR ставятся если фильм или тв сериал не оценен. Для них поставим тип unknown.

In [21]:
df['type'] = None
df['type'] = np.where(df['rating'].isin(['TV-14', 'TV-PG', 'TV-MA', 'TV-Y', 'TV-Y7-FV', 'TV-G', 'TV-Y7']), 'tv',
                      np.where(df['rating'].isin(['PG-13', 'R', 'PG', 'G']), 'movie', 'unknown'))
df.groupby('type').size()


type
movie      155
tv         334
unknown     11
dtype: int64

#### Разделение на возрастные группы по возрастному рейтингу

Далее также попробуем выделить возрастные группы для которых предназначены шоу, поделим их на три:
1. детские: G, TV-Y, TV-Y7, TV-G, TV-Y7-FV
2. семейные: PG, TV-PG
3. подростковые: PG-13, TV-14
4. взрослые: TV-MA, R
5. Без рейтинга: NR, UR

In [22]:
priznaki = pd.DataFrame({
    'rating': [
        'G','TV-G','TV-Y','TV-Y7','TV-Y7-FV',
        'PG','TV-PG',
        'PG-13','TV-14',
        'R','TV-MA','NR','UR'
        ],
    'audience_segment': [
        'Kids','Kids','Kids','Kids','Kids',
        'Family','Family',
        'Teens','Teens',
        'Adults','Adults','Adults','Adults'
        ],
    'kids_audience_fit': [
        1.0,1.0,1.0,1.0,1.0,
        0.7,0.7,
        0.2,0.2,
        0.0,0.0,0.0,0.0
        ],
    'youth_audience_fit': [
        0.2,0.2,0.2,0.2,0.2,
        0.7,0.7,
        1.0,1.0,
        0.4,0.4,0.4,0.4
        ],
    'adult_audience_fit': [
        0.2,0.2,0.2,0.2,0.2,
        0.4,0.4,
        0.6,0.6,
        1.0,1.0,1.0,1.0
        ]})

df = df.merge(priznaki,on='rating',how='left').reset_index()
df.head(2)

,level_0,index,title,rating,ratingLevel,release year,user rating score,user rating size,sceneDesc,type,audience_segment,kids_audience_fit,youth_audience_fit,adult_audience_fit
0,0,0,White Chicks,PG-13,Parents strongly cautioned. May be inappropria...,2004,82.0,80,"crude and sexual humor, language and some drug...",movie,Teens,0.2,1.0,0.6
1,1,1,Lucky Number Slevin,R,Restricted. May be inappropriate for children ...,2006,NaN,82,"strong violence, sexual content and adult lang...",movie,Adults,0.0,0.4,1.0


### Обогащение данными из внешних источников

#### Поиск фильмов с помощью API OMDB

In [23]:
import requests

def search_movie(row, title, year, key):
  """
  Парметры:
  movie - строка с информацией о фильме
  title - название столбца с названиями фильмов
  year - название столбца с годом выпуска
  Вывод:
  жанр, страна, рейтинг IMDB, количество оценивших, возрастная группа

  Выполняет поиск фильма по названию и году через OMDB API
  """

  url = "http://www.omdbapi.com/"
  params = {
    "apikey": key,
    "t": row[title],
    "type": "movie",
    "y": row[year]
    }
  try:
    m = requests.get(url, params=params).json()
    return {
      'genre': m.get("Genre"),
      'country': m.get("Country"),
      'imdbRating': m.get("imdbRating"),
      'imdbVotes': m.get("imdbVotes"),
      'rated': m.get("Rated")
      }
  except Exception as e:
    print(e)

def parsing_OMDB(df):
  df['all_OMDB'] = np.nan
  # парсинг данных
  df['all_OMDB'] = df[df['type'] == 'movie'].apply(
    search_movie,
    axis=1,
    title="title",
    year="release year",
    key=api_key,
    )
  # разделение списка на отдельные признаки
  temp = pd.json_normalize(df['all_OMDB'])
  df = pd.concat([df, temp], axis=1)
  # Дополнительная проверка на соответствие
  mask = (
    (df['type'] == 'movie') &
    (df['rated'] != df['rating']) &
    (df['rated'].notna())
    )
  cols = ['genre', 'country', 'imdbRating', 'imdbVotes', 'rated']
  df.loc[mask, cols] = None
  df.drop(columns=['all_OMDB'], inplace=True)
  return df



In [24]:
# Обогащаем датасет данными из OMDB (поиск по фильмам).
# Требует валидного API_KEY в .env и доступа в интернет.
df = parsing_OMDB(df)
df.head(2)

,level_0,index,title,rating,ratingLevel,release year,user rating score,user rating size,sceneDesc,type,audience_segment,kids_audience_fit,youth_audience_fit,adult_audience_fit,genre,country,imdbRating,imdbVotes,rated
0,0,0,White Chicks,PG-13,Parents strongly cautioned. May be inappropria...,2004,82.0,80,"crude and sexual humor, language and some drug...",movie,Teens,0.2,1.0,0.6,"Comedy, Crime",United States,5.9,"190,250",PG-13
1,1,1,Lucky Number Slevin,R,Restricted. May be inappropriate for children ...,2006,NaN,82,"strong violence, sexual content and adult lang...",movie,Adults,0.0,0.4,1.0,"Crime, Drama, Thriller","United Kingdom, Germany, Canada, United States",7.7,"337,898",R


In [25]:
# Просмотр пустых значений
df[df['type'] == 'movie'].isna().sum() / df[df['type'] == 'movie'].shape[0]

level_0               0.000000
index                 0.000000
title                 0.000000
rating                0.000000
ratingLevel           0.000000
release year          0.000000
user rating score     0.464516
user rating size      0.000000
sceneDesc             0.432258
type                  0.000000
audience_segment      0.000000
kids_audience_fit     0.000000
youth_audience_fit    0.000000
adult_audience_fit    0.000000
genre                 0.135484
country               0.135484
imdbRating            0.135484
imdbVotes             0.135484
rated                 0.135484
dtype: float64

Данный поиск проводится только по типу фильм, так как год выпуска телешоу в датасете указан неоднозначно, а для фильма год выпуска является точным значением. Парсинг проводится на основе `title, release year`, после проводится контрольная проверка по `rating`. При несоответсвии по одному из признаков информация удаляется.
Процент пропусков составил 13,5%, не найдено: 21 фильм.

### Кодирование признаков

Рассмотрим все категориальные признаки:

In [26]:
df.select_dtypes('object').info()


<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   title             500 non-null    object
 1   rating            500 non-null    str   
 2   ratingLevel       500 non-null    str   
 3   sceneDesc         89 non-null     object
 4   type              500 non-null    str   
 5   audience_segment  500 non-null    str   
 6   genre             134 non-null    str   
 7   country           134 non-null    str   
 8   imdbRating        134 non-null    str   
 9   imdbVotes         134 non-null    str   
 10  rated             134 non-null    str   
dtypes: object(2), str(9)
memory usage: 43.1+ KB


/var/folders/_c/gtb0wmg14r7ct0d6rnwqxg5w0000gn/T/ipykernel_45141/115827568.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes('object').info()


Классифицировать будем признаки: `rating, type, audience_segment`
Признак `ratingLevel` соответствует `rating` 1 к 1, его оставим как информативный, для нормализации в будущем можно вынести в словарь. `SceneDesc` мало заполнен и его использовать в дальнейшем не будем, но в ином случае кодировали бы его ключевыми словами. 
1. `rating` - рейтинг имеет 13 уникальных категорий и кодировать его напрямую нумерованием или OHE не целесообразно. Будем кодировать его нецелыми числами и в целую часть помещать похожие группы:
    - G, TV-G, TV-Y: 1.1, 1.2, 1.3
    - TV-Y7, TV-Y7-FV: 2.1, 2.2
    - PG, TV-PG: 3.1, 3.2
    - PG-13, TV-14: 4.1, 4.2
    - R, TV-MA: 5.1, 5.2
    - NR, UR: 0
2. `type` - кодируем через OHE
3. `audience_segment` - кодируем через OHE

In [27]:
code_rating = pd.DataFrame({
    'rating': [
        'G', 'TV-G', 'TV-Y',
        'TV-Y7', 'TV-Y7-FV',
        'PG', 'TV-PG',
        'PG-13', 'TV-14',
        'R', 'TV-MA',
        'NR', 'UR'
        ],
    'rating_num': [
        1.1, 1.2, 1.3,
        2.1, 2.2,
        3.1, 3.2,
        4.1, 4.2,
        5.1, 5.2,
        0, 0
    ]
})
df = df.merge(code_rating, on='rating')
df['type_cat'] = df['type']
df['audience_segment_cat'] = df['audience_segment']
df = pd.get_dummies(df, columns=['type', 'audience_segment'])
df.head()

,level_0,index,title,rating,ratingLevel,release year,user rating score,user rating size,sceneDesc,kids_audience_fit,...,rating_num,type_cat,audience_segment_cat,type_movie,type_tv,type_unknown,audience_segment_Adults,audience_segment_Family,audience_segment_Kids,audience_segment_Teens
0,0,0,White Chicks,PG-13,Parents strongly cautioned. May be inappropria...,2004,82.0,80,"crude and sexual humor, language and some drug...",0.2,...,4.1,movie,Teens,True,False,False,False,False,False,True
1,1,1,Lucky Number Slevin,R,Restricted. May be inappropriate for children ...,2006,NaN,82,"strong violence, sexual content and adult lang...",0.0,...,5.1,movie,Adults,True,False,False,True,False,False,False
2,2,2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80,None,0.2,...,4.2,tv,Teens,False,True,False,False,False,False,True
3,3,3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,80,None,0.2,...,4.2,tv,Teens,False,True,False,False,False,False,True
4,4,4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,80,None,0.7,...,3.2,tv,Family,False,True,False,False,True,False,False


### Сохранение чистого датасета

Сохраняем обработанный датафрейм в `NetflixShows_clear.csv` — этот файл загружается в ноутбуках с анализом.

In [28]:
# Сохраняем чистый датасет для дальнейшего анализа
df.to_csv('NetflixShows_clear.csv', index=False)
print('Сохранено:', df.shape[0], 'строк,', df.shape[1], 'столбцов')

Сохранено: 500 строк, 27 столбцов
